# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json). All dataset entities—record sets, fields, and columns—are referenced using their `@id` per the Croissant specification.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review all available record sets and their fields using their `@id`s. This step will help identify which record sets are available for extraction and analysis.

In [ ]:
# Explore record sets in the Croissant schema
record_sets_info = list(dataset.record_sets)

print("Available Record Sets:")
for rs in record_sets_info:
    print(f"- @id: {rs['@id']}  |  name: {rs.get('name', '[no name]')}")
    if 'fields' in rs and rs['fields']:
        for f in rs['fields']:
            print(f"    - field @id: {f['@id']}  |  name: {f.get('name', '[no name]')}")
    else:
        print("    (No fields found in this record set)")

## 3. Data Extraction
Load data from chosen record sets into DataFrames. Reference each record set and field using its Croissant `@id`.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets_info]

# Dictionary to hold DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} with {len(dataframes[record_set_id])} rows and columns: {list(dataframes[record_set_id].columns)}")
    else:
        print(f"Record set {record_set_id} yielded no records.")

# For illustration, pick the first available record set for EDA
if len(dataframes) > 0:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"\nSample data from record set {chosen_record_set_id}:")
    display(dataframes[chosen_record_set_id].head())
else:
    print("No record sets available for data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps using record set and field `@id`s. Typical steps include filtering, normalization, and grouping. All operations should directly reference fields by their canonical `@id`.

In [ ]:
# EDA: Filtering, normalization, and grouping by field `@id`
# Replace these with valid IDs and suitable numeric/group fields as revealed above
import numpy as np

if len(dataframes) > 0:
    df = dataframes[chosen_record_set_id]

    # Attempt to locate a numeric field for analysis
    # Choose the first field with numeric dtype
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_columns:
        print("No numeric fields found for EDA in this record set.")
    else:
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field '@id': {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_col = numeric_field_id + "_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-8)
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping: choose first non-numeric column
        group_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == 'object']
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} by {group_field_id} (top 5 groups):")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric grouping field found.")
else:
    print("No DataFrame to analyze.")

## 5. Visualization
Visualize distributions and relationships between fields using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_candidates:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated the use of `mlcroissant` to load, inspect, and perform EDA on the FAIR² dataset using Croissant entity `@id` references. Continue with advanced analyses as needed, always referencing data entities by their `@id` for clarity and consistency.